# 📊 Prática 2: Análise de Dados
### Workshop Dados na Prática · Nerd Data Lab + Programa AI

Os dados estão limpos. Agora é hora de **responder perguntas de negócio** usando **SQL**, a linguagem que aparece em quase toda vaga de dados.

A pergunta do dono do Café Tambaú continua a mesma: **por que as vendas caíram?**

Um bom analista não sai fazendo gráficos aleatórios. Ele parte de uma pergunta, busca a resposta nos dados e vai **refinando** até encontrar a causa.

**Como usar:** Shift + Enter para rodar. Nas células com ✏️, troque os `___`.

In [ ]:
# ⚙️ Configuração: rode esta célula primeiro (Shift + Enter)
import os
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 30)
plt.rcParams["figure.figsize"] = (10, 4)

# De onde vêm os dados:
# 1) Se você subiu a pasta cafe_tambau para o Colab (ícone de pasta à esquerda), ela é usada.
# 2) Se não, os arquivos são baixados deste endereço.
URL_BASE = "https://raw.githubusercontent.com/wuldson-franco/workshop_dados_pratica/main/"
PASTA_LOCAL = "./"

def ler(arquivo, **opcoes):
    caminho = PASTA_LOCAL + arquivo
    if os.path.exists(caminho):
        return pd.read_csv(caminho, **opcoes)
    return pd.read_csv(URL_BASE + arquivo, **opcoes)

print("Tudo pronto! ✅")

import duckdb

def sql(consulta):
    return duckdb.sql(consulta).df()

In [ ]:
vendas = ler("dados_tratados/vendas_limpas.csv", parse_dates=["data_hora"])
clientes = ler("dados_tratados/clientes_limpos.csv")
avaliacoes = ler("dados_tratados/avaliacoes.csv", parse_dates=["data_avaliacao"])

sql("SELECT * FROM vendas LIMIT 5")

## 📖 Mini guia de SQL

```sql
SELECT coluna, SUM(valor)     -- o que eu quero ver
FROM tabela                   -- de onde vêm os dados
WHERE condição                -- filtro
GROUP BY coluna               -- agrupar
ORDER BY coluna               -- ordenar
```

## 1. As vendas caíram mesmo? Faturamento por mês

In [ ]:
# ✏️ COMPLETE: troque os ___ pelo código certo
fat_mes = sql("""
    SELECT mes, ROUND(SUM(valor_total), 2) AS faturamento
    FROM vendas
    GROUP BY ___
    ORDER BY mes
""")
fat_mes

In [ ]:
fat_mes.plot(x="mes", y="faturamento", kind="bar", legend=False, color="#1FD3F0", title="Faturamento por mês (R$)")
plt.show()

E se a gente olhasse por **trimestre**, como muitos relatórios fazem?

In [ ]:
sql("""
    SELECT trimestre, ROUND(SUM(valor_total), 2) AS faturamento
    FROM vendas
    GROUP BY trimestre
    ORDER BY trimestre
""")

🤔 **Para pensar:** por trimestre, a queda parece pequena. Por mês, agosto e setembro caíram bem mais. Julho é férias em João Pessoa, e o movimento dos turistas **escondeu** o problema.

💡 Lição: **o nível de detalhe muda a conclusão.**

## 2. Caiu o número de pedidos ou o valor de cada pedido?

In [ ]:
# ✏️ COMPLETE: troque os ___ pelo código certo
sql("""
    SELECT mes,
           COUNT(DISTINCT ___) AS pedidos,
           ROUND(SUM(valor_total) / COUNT(DISTINCT id_venda), 2) AS ticket_medio
    FROM vendas
    GROUP BY mes
    ORDER BY mes
""")

✅ O ticket médio quase não mudou. **O problema é que menos pedidos estão acontecendo.** Mas em qual momento do dia?

## 3. Em que período do dia os pedidos caíram?

In [ ]:
periodo = sql("""
    SELECT trimestre, periodo, COUNT(DISTINCT id_venda) AS pedidos
    FROM vendas
    GROUP BY trimestre, periodo
""").pivot(index="periodo", columns="trimestre", values="pedidos")

periodo["variação T3 x T2 (%)"] = ((periodo["T3"] / periodo["T2"] - 1) * 100).round(1)
periodo

In [ ]:
periodo[["T1", "T2", "T3"]].plot(kind="bar", title="Pedidos por período do dia", color=["#2a4a7a", "#1FD3F0", "#3DF29A"])
plt.xticks(rotation=0)
plt.show()

## 4. Algum produto caiu mais que os outros?

In [ ]:
# ✏️ COMPLETE: troque os ___ pelo código certo
produtos_tri = sql("""
    SELECT nome_produto, trimestre, SUM(quantidade) AS unidades
    FROM vendas
    WHERE trimestre IN ('T2', '___')
    GROUP BY nome_produto, trimestre
""").pivot(index="nome_produto", columns="trimestre", values="unidades")

produtos_tri["variação (%)"] = ((produtos_tri["T3"] / produtos_tri["T2"] - 1) * 100).round(1)
produtos_tri.sort_values("variação (%)")

Um produto despencou. O que aconteceu com o **preço** dele?

In [ ]:
sql("""
    SELECT mes, ROUND(AVG(preco_unitario), 2) AS preco_medio, SUM(quantidade) AS unidades
    FROM vendas
    WHERE nome_produto = 'Cappuccino'
    GROUP BY mes
    ORDER BY mes
""")

## 5. E o delivery?

In [ ]:
# ✏️ COMPLETE: troque os ___ pelo código certo
sql("""
    SELECT mes,
           COUNT(DISTINCT id_venda) AS pedidos_ifood,
           ROUND(AVG(tempo_entrega_min), 1) AS tempo_medio_entrega
    FROM vendas
    WHERE canal = '___'
    GROUP BY mes
    ORDER BY mes
""")

## 6. O que os clientes estão dizendo? Nota média por mês

In [ ]:
notas = sql("""
    SELECT MONTH(data_avaliacao) AS mes, ROUND(AVG(nota), 2) AS nota_media, COUNT(*) AS avaliacoes
    FROM avaliacoes
    WHERE MONTH(data_avaliacao) <= 9
    GROUP BY mes
    ORDER BY mes
""")
notas.plot(x="mes", y="nota_media", marker="o", legend=False, color="#3DF29A", title="Nota média das avaliações")
plt.ylim(1, 5)
plt.show()
notas

## 7. Quem parou de vir? Juntando tabelas com JOIN

As vendas têm o `id_cliente`, e a tabela de clientes tem o **bairro**. Para cruzar as duas, usamos `JOIN`.

Vamos contar quantos clientes diferentes compraram em cada bairro em **junho** (antes dos problemas) e em **setembro**.

In [ ]:
# ✏️ COMPLETE: troque os ___ pelo código certo
bairros = sql("""
    SELECT c.bairro, v.mes, COUNT(DISTINCT v.id_cliente) AS clientes
    FROM vendas v
    JOIN clientes c ON v.id_cliente = c.___
    WHERE v.mes IN (6, 9)
    GROUP BY c.bairro, v.mes
""").pivot(index="bairro", columns="mes", values="clientes")

bairros["variação (%)"] = ((bairros[9] / bairros[6] - 1) * 100).round(1)
bairros.sort_values("variação (%)")

🤔 Os bairros que mais perderam clientes ficam todos **na orla**. E foi justamente em julho que um café concorrente abriu por lá...

## ⭐ Desafio livre

Escreva a sua própria consulta! Algumas ideias: qual a forma de pagamento mais usada? Qual produto dá mais **lucro**? Qual o dia da semana mais movimentado?

In [ ]:
# ✏️ COMPLETE: troque os ___ pelo código certo
sql("""
    SELECT ___
    FROM vendas
    ___
""")

## 📈 Bônus: monte um dashboard

1. Baixe o arquivo `vendas_limpas.csv` e importe no **Google Sheets**.
2. Abra o **Looker Studio** (lookerstudio.google.com) e crie um relatório conectado à planilha.
3. Monte 3 gráficos: faturamento por mês, pedidos por período do dia e pedidos por canal.

## 🎯 Seu insight

Escreva abaixo, em uma ou duas frases, **o que você descobriu e o que recomendaria ao dono do café**:

> ✍️ *Escreva aqui...*

📋 Envie seu insight no formulário para ganhar seu cupom do sorteio! 🎁